In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install catboost

In [ ]:

# Kaggle Playground Series S6E2 – CatBoost Fixed Submission


import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

# GPU auto-detection
try:
    import cupy
    GPU_ACC = True
except Exception:
    GPU_ACC = False

print(f"GPU enabled: {GPU_ACC}")


# Paths

TRAIN_PATH = "/kaggle/input/playground-series-s6e2/train.csv"
TEST_PATH = "/kaggle/input/playground-series-s6e2/test.csv"
SUB_PATH = "/kaggle/input/playground-series-s6e2/sample_submission.csv"
OUT_PATH = "/kaggle/working/submission.csv"  


# Load data

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUB_PATH)

TARGET = "Heart Disease"

X = train.drop(columns=[TARGET])
y = train[TARGET]


# Feature groups

cat_cols = [
    'Sex', 'Chest pain type', 'FBS over 120', 'EKG results',
    'Exercise angina', 'Slope of ST',
    'Number of vessels fluro', 'Thallium'
]
num_cols = [c for c in X.columns if c not in cat_cols]


# Preprocessing

# Numeric: median imputation
num_imputer = SimpleImputer(strategy="median")
X[num_cols] = num_imputer.fit_transform(X[num_cols])
test[num_cols] = num_imputer.transform(test[num_cols])

# Categoricals as strings (CatBoost-native)
X[cat_cols] = X[cat_cols].astype(str)
test[cat_cols] = test[cat_cols].astype(str)


# CatBoost model

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_state=42,
    task_type="GPU" if GPU_ACC else "CPU",
    verbose=False
)


# Cross-validation

FOLDS = 5
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model.fit(
        X_tr, y_tr,
        cat_features=cat_cols
    )

    oof_preds[va_idx] = model.predict_proba(X_va)[:, 1]
    test_preds += model.predict_proba(test)[:, 1] / FOLDS

    fold_auc = roc_auc_score(y_va, oof_preds[va_idx])
    print(f"Fold {fold} AUC: {fold_auc:.5f}")

# Overall CV score
cv_auc = roc_auc_score(y, oof_preds)
print(f"\nOverall CV AUC: {cv_auc:.5f}")


# Submission

# Assign probabilities to submission column
submission.iloc[:, 1] = test_preds.astype(float)

# Save to working directory
submission.to_csv(OUT_PATH, index=False)
print(f"\n✅ submission.csv saved at {OUT_PATH}")

submission.head()
